In [6]:
from KG.kg import ingest_Chunks, create_nodes, create_relationship, create_vector_index, embed_text
from KG.chunking import split_data_from_file
from KG.config import load_neo4j_graph
from KG.backup import wipe_graph
import json
graph, gemini_api, _ = load_neo4j_graph()

In [7]:
file_names = [
    "Book_1_Philosopher_s_Stone",
    "Book_2_Chamber_of_Secrets",
    "Book_3_Prisoner_of_Azkaban",
    "Book_4_Goblet_of_Fire",
    "Book_5_Order_of_the_Phoenix",
    "Book_6_Half_Blood_Prince",
    "Book_7_Deathly_Hallows",
]

In [8]:
for i, name in enumerate(file_names, 1):
    print(f"\n=== [{i}/{len(file_names)}] Ingesting {name} ===")
    file = f"data/{name}.json"
    chunks = split_data_from_file(file)
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    create_nodes(graph=graph, data=data, node_label="Book", node_name=name)
    ingest_Chunks(graph=graph, chunks=chunks, node_name=name, node_label='Chunk')
    print(f"=== [{i}/{len(file_names)}] Done: {name} ===")


=== [1/7] Ingesting Book_1_Philosopher_s_Stone ===
['chap-1', 'chap-2', 'chap-3', 'chap-4', 'chap-5', 'chap-6', 'chap-7', 'chap-8', 'chap-9', 'chap-10', 'chap-11', 'chap-12', 'chap-13', 'chap-14', 'chap-15', 'chap-16', 'chap-17']
Processing chap-1 from data/Book_1_Philosopher_s_Stone.json
	Split into 15 chunks
Processing chap-2 from data/Book_1_Philosopher_s_Stone.json
	Split into 11 chunks
Processing chap-3 from data/Book_1_Philosopher_s_Stone.json
	Split into 12 chunks
Processing chap-4 from data/Book_1_Philosopher_s_Stone.json
	Split into 12 chunks
Processing chap-5 from data/Book_1_Philosopher_s_Stone.json
	Split into 21 chunks
Processing chap-6 from data/Book_1_Philosopher_s_Stone.json
	Split into 20 chunks
Processing chap-7 from data/Book_1_Philosopher_s_Stone.json
	Split into 14 chunks
Processing chap-8 from data/Book_1_Philosopher_s_Stone.json
	Split into 10 chunks
Processing chap-9 from data/Book_1_Philosopher_s_Stone.json
	Split into 16 chunks
Processing chap-10 from data/Bo

In [11]:
rel_section_chunk = """
MATCH (s:Section), (c:Chunk)
WHERE s.type = c.source AND s.parent_name = c.node_name
MERGE (s)-[:HAS_CHUNK]->(c);
"""

rel_book_section = """
MATCH (b:Book), (s:Section)
WHERE b.name = s.parent_name
MERGE (b)-[:HAS_SECTION]->(s);
"""

queries = [rel_section_chunk, rel_book_section]

for query in queries:
    create_relationship(graph=graph, query=query)

[#CDF7]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.161.242', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 1.058757862920014s (Unable to retrieve routing information)
[#CDFE]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-16-0113.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Transaction failed and will be retried in 2.309015864635977s (Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-16-0113.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))))


In [12]:
create_vector_index(graph=graph, index_name='Chunk')

In [13]:
from KG.entities import extract_entities

for i, name in enumerate(file_names, 1):
    print(f"\n=== [{i}/{len(file_names)}] Extracting entities: {name} ===")
    extract_entities(graph=graph, api_key=gemini_api, batch_size=15, book_filter=name)


=== [1/7] Extracting entities: Book_1_Philosopher_s_Stone ===
Extracting entities from 251 chunks.


Extracting: 100%|███████████████████████████████████████████████████| 17/17 [13:35<00:00, 47.94s/it]


Finished entity extraction.

=== [2/7] Extracting entities: Book_2_Chamber_of_Secrets ===
Extracting entities from 281 chunks.


Extracting: 100%|███████████████████████████████████████████████████| 19/19 [16:31<00:00, 52.17s/it]


Finished entity extraction.

=== [3/7] Extracting entities: Book_3_Prisoner_of_Azkaban ===
Extracting entities from 348 chunks.


...

  retry in 8s (Expecting property name enclosed in double quotes: line 200 column 5 (char 8042))


Extracting: 100%|███████████████████████████████████████████████████| 38/38 [28:47<00:00, 45.46s/it]


Finished entity extraction.

=== [7/7] Extracting entities: Book_7_Deathly_Hallows ===
Extracting entities from 638 chunks.


Extracting: 100%|███████████████████████████████████████████████████| 43/43 [33:19<00:00, 46.50s/it]

Finished entity extraction.


In [3]:
for i, name in enumerate(file_names, 1):
    print(f"\n=== [{i}/{len(file_names)}] Embedding {name} ===")
    embed_text(graph=graph, api_key=gemini_api, node_name='Chunk', batch_size=48, book_filter=name)


=== [1/7] Embedding Book_1_Philosopher_s_Stone ===
Starting embedding update...
Found 0 nodes without embeddings.


Embedding: 0it [00:00, ?it/s]

Finished embedding update.

=== [2/7] Embedding Book_2_Chamber_of_Secrets ===


Starting embedding update...
Found 0 nodes without embeddings.


Embedding: 0it [00:00, ?it/s]

...

Embedding: 0it [00:00, ?it/s]

Finished embedding update.

=== [7/7] Embedding Book_7_Deathly_Hallows ===


Starting embedding update...
Found 0 nodes without embeddings.


Embedding: 0it [00:00, ?it/s]

Finished embedding update.


In [16]:
from KG.normalize_relationships import normalize_relationships

mapping = normalize_relationships(graph=graph, api_key=gemini_api)

[#E6B1]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.161.242', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 0.8556803089304527s (Unable to retrieve routing information)
[#FC19]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-16-0113.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Transaction failed and will be retried in 1.753060907399644s (Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-16-0113.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))))


Found 558 non-structural relationship types to normalize.
Rewrote 433 relationship types to their canonical form.


In [18]:
from KG.normalize_entities import normalize_entities

entity_mapping = normalize_entities(graph=graph, api_key=gemini_api)

[#D636]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.161.242', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 0.9931220488319912s (Unable to retrieve routing information)
[#E121]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-16-0113.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Transaction failed and will be retried in 1.9770102058754426s (Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-16-0113.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))))


Checking top 400 entities for alias duplicates.
Merged 172 alias entities into their canonical form.


In [21]:
from google import genai

In [22]:
from KG.entities import _call_gemini_json

null_type = graph.query("MATCH (e:Entity) WHERE e.type IS NULL RETURN e.name AS name")
names_block = "\n".join(e["name"] for e in null_type)

prompt = f"""Classify each of these Harry Potter entity names into exactly one type:
Character, Location, Spell, Object, Creature, or Organization.
Return ONLY a JSON object mapping each name to its type.

Names:
{names_block}
"""
type_mapping = _call_gemini_json(genai.Client(api_key=gemini_api), "gemini-3.5-flash-lite", prompt)

for name, etype in type_mapping.items():
    graph.query("MATCH (e:Entity {name: $name}) SET e.type = $type", params={"name": name, "type": etype})

In [27]:
result = graph.query("MATCH (e:Entity) WHERE e.type IS NULL RETURN count(e) AS count")
print(result)

[{'count': 0}]


In [24]:
graph.query("MATCH (e:Entity {name: 'TARGET_OF'}) DETACH DELETE e")

[]

In [25]:
result = graph.query("MATCH (e:Entity) WHERE e.type IS NULL RETURN e.name")
print(result)

[{'e.name': "Harry's dad"}]


In [26]:
graph.query("MATCH (e:Entity {name: \"Harry's dad\"}) SET e.type = 'Character'")

[]